In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

def perform_eda_and_cleaning(df):
    print("\n--- 🔍 STARTING EDA & DATA CLEANING ---")
    
    print(f"Original Dataset Shape: {df.shape}")
    
    duplicates = df.duplicated().sum()
    df = df.drop_duplicates()
    print(f"Dropped {duplicates} duplicate rows.")
    
    initial_nulls = df[['views', 'likes', 'comment_count']].isnull().sum().sum()
    df = df.dropna(subset=['views', 'likes', 'comment_count'])
    print(f"Dropped {initial_nulls} rows with missing essential features.")
    print(f"Cleaned Dataset Shape: {df.shape}")
    
    if 'views_per_hour' not in df.columns:
        df['views_per_hour'] = df['views'] / 24 
        
    if 'is_viral' not in df.columns:
        df['is_viral'] = (df['views_per_hour'] >= 5000).astype(int)
        
    print("\n--- 📊 EDA STATISTICS ---")
    print(df[['likes', 'comment_count', 'views_per_hour']].describe())
    
    print(f"\nTarget Variable Distribution (0=Normal, 1=Viral):\n{df['is_viral'].value_counts()}")
    print("--- ✅ EDA & CLEANING COMPLETE ---\n")
    
    return df

def train_pipeline(csv_path):
    print(f"Loading historical data from {csv_path}...")
    raw_df = pd.read_csv(csv_path)
    
    df = perform_eda_and_cleaning(raw_df)
    
    # --- 🚨 DATA LEAKAGE FIXED HERE 🚨 ---
    # Removed 'views_per_hour' from input features
    features = ['likes', 'comment_count']
    X = df[features]
    y = df['is_viral']
    
    print("Splitting dataset into train and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print("Scaling features to prevent data leakage...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print("Training Random Forest Classifier...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n--- 🎯 MODEL PERFORMANCE REPORT ---")
    print(f"Model Accuracy (Realistic): {acc * 100:.2f}%")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    
    os.makedirs('../models', exist_ok=True)
    joblib.dump(model, '../models/viral_model.pkl')
    joblib.dump(scaler, '../models/scaler.pkl')
    print("\nSUCCESS: Model and Scaler have been successfully saved to the '../models/' directory! ✅")

if __name__ == '__main__':
    train_pipeline('../data/preprocessed_youtube_data.csv')

Loading historical data from ../data/preprocessed_youtube_data.csv...

--- 🔍 STARTING EDA & DATA CLEANING ---
Original Dataset Shape: (161470, 18)
Dropped 0 duplicate rows.
Dropped 0 rows with missing essential features.
Cleaned Dataset Shape: (161470, 18)

--- 📊 EDA STATISTICS ---
              likes  comment_count  views_per_hour
count  1.614700e+05   1.614700e+05    1.614700e+05
mean   6.566194e+04   7.035494e+03    1.008273e+05
std    2.260617e+05   3.404121e+04    4.348953e+05
min    0.000000e+00   0.000000e+00    9.291667e+00
25%    1.975000e+03   2.790000e+02    4.230760e+03
50%    9.840000e+03   1.144000e+03    1.603081e+04
75%    4.006275e+04   4.144750e+03    5.581367e+04
max    5.613827e+06   1.626501e+06    1.768912e+07

Target Variable Distribution (0=Normal, 1=Viral):
is_viral
1    116973
0     44497
Name: count, dtype: int64
--- ✅ EDA & CLEANING COMPLETE ---

Splitting dataset into train and test sets...
Scaling features to prevent data leakage...
Training Random Forest 